**CSI 4106 Introduction to Artificial Intelligence**  
*Assignment 2: Mission Interruption Prediction with Classical Models*  
*Due: November 2, 2026, at 11 PM*

# Identification

Name: John Lopes  
Student number: 300319245  
Report title: Mission Interruption Prediction with Classical Models

# 1. Data and objective

The objective is to predict whether an autonomous underwater vehicle
returns before completing its planned survey. The positive class is
`mission_interrupted=1`.

- [Assignment 2 data
  directory](https://github.com/turcotte/csi4106-f26/tree/main/assignments-data/a2)
- [asrc_missions_complete.csv](https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_complete.csv)
- [asrc_missions_cleaned.csv](https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_cleaned.csv)
- [asrc_missions_hidden.csv](https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_hidden.csv)

Consult the assignment handout for the data relationships, experimental
constraints, required grids, and submission format.

# 2. Classical machine-learning study

## 2.1 Load the data and establish the experimental protocol

Load the complete and cleaned labelled datasets. Local copies may be
reused, but the notebook must obtain an unavailable source file from its
supplied URL without a manual upload. Report their shapes and the first
five complete rows. Verify that their columns, row order, and response
correspond. Separate the complete predictors and response, then create
the required stratified 80/20 split with `random_state=4106`. Report
class counts and proportions for the complete data and both partitions.

In [6]:
from sklearn.model_selection import train_test_split
# adapted from scikit-learn documentation [1]
import pandas as pd
import numpy as np

ogcomplete = pd.read_csv(f"https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_complete.csv")
ogcleaned = pd.read_csv(f"https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_cleaned.csv")
oghidden = pd.read_csv(f"https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a2/asrc_missions_hidden.csv")
completedf = ogcomplete.copy()
cleanedf = ogcleaned.copy()
hiddendf = oghidden.copy()
random_state = 4106
print("complete asrc missions: " + str(completedf.shape))
print("cleaned asrc missions: " + str(cleanedf.shape) + "\n")

print("complete asrc missions:")
print(completedf.head())
print("\n \n cleaned asrc missions:")
print(cleanedf.head())

print("\n \n columns correspond: " + str(completedf.columns.equals(cleanedf.columns)))
print("row order correspond: " + str(completedf.index.equals(cleanedf.index)))
print("response values correspond: " + str(completedf["mission_interrupted"].equals(cleanedf["mission_interrupted"])))

inputdf = completedf.drop(columns=["mission_interrupted"])
outputdf = completedf["mission_interrupted"]
xtrain, xtest, ytrain, ytest = train_test_split(inputdf, outputdf, test_size=0.2, random_state=random_state, stratify=outputdf)

print("complete df " + str(completedf["mission_interrupted"].value_counts()))
print("train df " + str(ytrain.value_counts()))
print("test df " + str(ytest.value_counts()))

complete asrc missions: (6371, 13)
cleaned asrc missions: (6371, 13)

complete asrc missions:
   planned_depth_m  descent_current_speed_m_s  water_temperature_c  \
0           1400.0                       0.55                  3.3   
1           2700.0                       0.68                  3.0   
2           2870.0                       0.15                  3.6   
3           2840.0                       0.39                  2.5   
4           2430.0                       0.34                  2.8   

   sonar_noise_db  navigation_drift_m  battery_health_pct  \
0            66.0                10.4                87.0   
1            78.5                49.9                83.0   
2            78.7                 5.8                92.0   
3            77.3                54.6                89.0   
4            77.8                 8.7                89.0   

   thruster_vibration_mm_s  missions_since_service vehicle_class  \
0                      3.1                      14

**Checks and interpretation:**

in this step the completed file is checked with the cleaned file with a equals() method to ensure that they have the same info when it comes to columns, row order and response value. two new dataframes were created as a copy of the completed file except one of these dataframes contains only the mission_interupted column and the other with every column but mission_interupted.Finally a new set of dataframes was also created to allow for model training on 80% of the data and testing of the model post training on 20% of the data

## 2.2 Construct leakage-safe preprocessing pipelines

Identify the numerical, nominal, ordinal, and binary predictors and
present the proposed preprocessing in a compact table. Construct the
required `ColumnTransformer` and model pipelines. Explain the choices,
how later data with an otherwise valid category absent from a training
fold can be transformed, and how the design prevents data leakage during
cross-validation.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
# adapted from scikit-learn documentation [1]

data = { "planned_depth_m": ["numerical", "standard scaling"], "descent_current_speed_m_s": ["numerical", "standard scaling"], "water_temperature_c": ["numerical", "standard scaling"], "sonar_noise_db": ["numerical", "standard scaling"], "navigation_drift_m": ["numerical", "standard scaling"], "battery_health_pct": ["numerical", "standard scaling"], "thruster_vibration_mm_s": ["numerical", "standard scaling"], "missions_since_service": ["numerical", "standard scaling"], "vehicle_class": ["nominal", "one-hot encoding"], "seabed_terrain": ["nominal", "one-hot encoding"], "maintenance_grade": ["ordinal", "ordinal encoding"], "firmware_current": ["binary", "none"] }
print(pd.DataFrame(data).head())

ct = ColumnTransformer(transformers=[("numerical", StandardScaler(), ["planned_depth_m", "descent_current_speed_m_s", "water_temperature_c", "sonar_noise_db", "navigation_drift_m", "battery_health_pct", "thruster_vibration_mm_s", "missions_since_service"]), ("nominal", OneHotEncoder(handle_unknown="ignore"), ["vehicle_class", "seabed_terrain"]), ("ordinal", OrdinalEncoder(categories=[["poor","fair","good","excellent"]]), ["maintenance_grade"])], remainder="passthrough")
processingpipe = Pipeline(steps=[("preprocess", ct)])
processingpipe.fit(xtrain)

    planned_depth_m descent_current_speed_m_s water_temperature_c  \
0         numerical                 numerical           numerical   
1  standard scaling          standard scaling    standard scaling   

     sonar_noise_db navigation_drift_m battery_health_pct  \
0         numerical          numerical          numerical   
1  standard scaling   standard scaling   standard scaling   

  thruster_vibration_mm_s missions_since_service     vehicle_class  \
0               numerical              numerical           nominal   
1        standard scaling       standard scaling  one-hot encoding   

     seabed_terrain maintenance_grade firmware_current  
0           nominal           ordinal           binary  
1  one-hot encoding  ordinal encoding             none  


,steps,"[('preprocess', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numerical', ...), ('nominal', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


**Preprocessing rationale:**

## 2.3 Evaluate baselines and default models

Evaluate the always-positive baseline, the three classifiers studied in
class, and the required random-forest extension using the same folds.
Report mean and standard deviation for precision, recall, F1, and ROC
AUC. Consult the handout for the short introduction to random forest.

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
# adapted from scikit-learn documentation [1]

baseline = DummyClassifier(strategy="constant", constant=1)
baseline.fit(xtrain, ytrain)


logisticpipe = Pipeline(steps=[("preprocess", ct), ("classifier", LogisticRegression(random_state=random_state))])
decisionpipe = Pipeline(steps=[("preprocess", ct), ("classifier", DecisionTreeClassifier(random_state=random_state))])
neighbourpipe = Pipeline(steps=[("preprocess", ct), ("classifier", KNeighborsClassifier())])
forestpipe = Pipeline(steps=[("preprocess", ct), ("classifier", RandomForestClassifier(random_state=random_state))])

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
scoring = {"precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}

base = cross_validate(baseline, xtrain, ytrain, cv=folds, scoring=scoring)
logistic = cross_validate(logisticpipe, xtrain, ytrain, cv=folds, scoring=scoring)
decision = cross_validate(decisionpipe, xtrain, ytrain, cv=folds, scoring=scoring)
neighbour = cross_validate(neighbourpipe, xtrain, ytrain, cv=folds, scoring=scoring)
forest = cross_validate(forestpipe, xtrain, ytrain, cv=folds, scoring=scoring)

print("baseline:")
print("mean precision: " + str(np.mean(base["test_precision"])) + "\nstd precision: " + str(np.std(base["test_precision"])) + "\nmean recall: " + str(np.mean(base["test_recall"])) + "\nstd recall: " + str(np.std(base["test_recall"])) + "\nmean F1: " + str(np.mean(base["test_f1"])) + "\nstd F1: " + str(np.std(base["test_f1"])) + "\nmean ROC AUC: " + str(np.mean(base["test_roc_auc"])) + "\nstd ROC AUC: " + str(np.std(base["test_roc_auc"])))
print("\nlogistic regression:")
print("mean precision: " + str(np.mean(logistic["test_precision"])) + "\nstd precision: " + str(np.std(logistic["test_precision"])) + "\nmean recall: " + str(np.mean(logistic["test_recall"])) + "\nstd recall: " + str(np.std(logistic["test_recall"])) + "\nmean F1: " + str(np.mean(logistic["test_f1"])) + "\nstd F1: " + str(np.std(logistic["test_f1"])) + "\nmean ROC AUC: " + str(np.mean(logistic["test_roc_auc"])) + "\nstd ROC AUC: " + str(np.std(logistic["test_roc_auc"])))
print("\ndecision tree:")
print("mean precision: " + str(np.mean(decision["test_precision"])) + "\nstd precision: " + str(np.std(decision["test_precision"])) + "\nmean recall: " + str(np.mean(decision["test_recall"])) + "\nstd recall: " + str(np.std(decision["test_recall"])) + "\nmean F1: " + str(np.mean(decision["test_f1"])) + "\nstd F1: " + str(np.std(decision["test_f1"])) + "\nmean ROC AUC: " + str(np.mean(decision["test_roc_auc"])) + "\nstd ROC AUC: " + str(np.std(decision["test_roc_auc"])))
print("\nK-nearest neighbors:")
print("mean precision: " + str(np.mean(neighbour["test_precision"])) + "\nstd precision: " + str(np.std(neighbour["test_precision"])) + "\nmean recall: " + str(np.mean(neighbour["test_recall"])) + "\nstd recall: " + str(np.std(neighbour["test_recall"])) + "\nmean F1: " + str(np.mean(neighbour["test_f1"])) + "\nstd F1: " + str(np.std(neighbour["test_f1"])) + "\nmean ROC AUC: " + str(np.mean(neighbour["test_roc_auc"])) + "\nstd ROC AUC: " + str(np.std(neighbour["test_roc_auc"])))
print("\nrandom forest:")
print("mean precision: " + str(np.mean(forest["test_precision"])) + "\nstd precision: " + str(np.std(forest["test_precision"])) + "\nmean recall: " + str(np.mean(forest["test_recall"])) + "\nstd recall: " + str(np.std(forest["test_recall"])) + "\nmean F1: " + str(np.mean(forest["test_f1"])) + "\nstd F1: " + str(np.std(forest["test_f1"])) + "\nmean ROC AUC: " + str(np.mean(forest["test_roc_auc"])) + "\nstd ROC AUC: " + str(np.std(forest["test_roc_auc"])))

baseline:
mean precision: 0.32594181146452694
std precision: 0.0002643883853836071
mean recall: 1.0
std recall: 0.0
mean F1: 0.49163812302950427
std F1: 0.00030067240985982564
mean ROC AUC: 0.5
std ROC AUC: 0.0

logistic regression:
mean precision: 0.7383404096860715
std precision: 0.03542721040613116
mean recall: 0.5261767791888273
std recall: 0.02126295683271736
mean F1: 0.6142181253581045
std F1: 0.02378671360308385
mean ROC AUC: 0.7866702131183638
std ROC AUC: 0.009944708889175754

decision tree:
mean precision: 0.5990608361850828
std precision: 0.008841623846829235
mean recall: 0.6219002134664786
std recall: 0.012685453986283754
mean F1: 0.6101267490753426
std F1: 0.005523583464754055
mean ROC AUC: 0.7102223046953936
std ROC AUC: 0.004014522724581193

K-nearest neighbors:
mean precision: 0.7639422744881537
std precision: 0.018432643892246234
mean recall: 0.569517710481566
std recall: 0.02549109763699665
mean F1: 0.6523983834329521
std F1: 0.02228382927634915
mean ROC AUC: 0.822659

**Baseline and default-model analysis:**

## 2.4 Tune the models

Perform the required grid searches on the development partition. For
each model, report its best configuration and cross-validation metrics,
compare training and validation F1, and use compact tables or figures
that collectively show the effect of every searched hyperparameter.
Every interaction need not be displayed.

### Logistic regression

In [24]:
from sklearn.model_selection import GridSearchCV
# adapted from scikit-learn documentation [1]

logisticsearch = GridSearchCV(estimator=Pipeline(steps=[("preprocess", ct), ("classifier", LogisticRegression(random_state=random_state))]), param_grid={"classifier__C": [0.001, 0.01, 0.1, 1, 10], "classifier__class_weight": [None, "balanced"]}, scoring=scoring, cv=folds, refit="f1")

logisticsearch.fit(xtrain, ytrain)

print("Best parameters:", logisticsearch.best_params_)
print("Best mean CV F1:", logisticsearch.best_score_)

print("precision mean: " + str(logisticsearch.cv_results_["mean_test_precision"][logisticsearch.best_index_]) + " precision std:" + str(logisticsearch.cv_results_["std_test_precision"][logisticsearch.best_index_]))
print("recall mean: " + str(logisticsearch.cv_results_["mean_test_recall"][logisticsearch.best_index_]) + " recall std:" + str(logisticsearch.cv_results_["std_test_recall"][logisticsearch.best_index_]))
print("f1 mean: " + str(logisticsearch.cv_results_["mean_test_f1"][logisticsearch.best_index_]) + " f1 std:" + str(logisticsearch.cv_results_["std_test_f1"][logisticsearch.best_index_]))
print("roc_auc mean: " + str(logisticsearch.cv_results_["mean_test_roc_auc"][logisticsearch.best_index_]) + " roc_auc std:" + str(logisticsearch.cv_results_["std_test_roc_auc"][logisticsearch.best_index_]))

resultsdf = pd.DataFrame(logisticsearch.cv_results_)

firstresult = resultsdf[(resultsdf["param_classifier__C"] == 0.001) & (resultsdf["param_classifier__class_weight"] == None)]
print(str(firstresult["mean_test_f1"].iloc[0]) + " test")

secondresult = resultsdf[(resultsdf["param_classifier__C"] == 1) & (resultsdf["param_classifier__class_weight"] == "balanced")]
print(str(secondresult["mean_test_f1"].iloc[0]) + " test")

thirdresult = resultsdf[(resultsdf["param_classifier__C"] == 10) & (resultsdf["param_classifier__class_weight"] == "balanced")]
print(str(thirdresult["mean_test_f1"].iloc[0]) + " test")


Best parameters: {'classifier__C': 1, 'classifier__class_weight': 'balanced'}
Best mean CV F1: 0.6319388425506288
precision mean: 0.5813732668545618 precision std:0.014167151671170084
recall mean: 0.6923495784941568 recall std:0.009155186310488353
f1 mean: 0.6319388425506288 f1 std:0.010216486312560046
roc_auc mean: 0.788147588847331 roc_auc std:0.00934866408064986


IndexError: single positional indexer is out-of-bounds

**Analysis:**

### Decision tree

In [10]:
decisionsearch = GridSearchCV(estimator=Pipeline(steps=[("preprocess", ct), ("classifier", DecisionTreeClassifier(random_state=random_state))]), param_grid={"max_depth": [3, 7, 12, 20, None], "min_samples_leaf": [1, 5, 10, 20, 50]}, scoring="f1", cv=folds, refit=True)

decisionsearch.fit(xtrain, ytrain)

print("Best configuration:", decisionsearch.best_params_)
print("Best mean CV F1:", decisionsearch.best_score_)

ValueError: Invalid parameter 'max_depth' for estimator Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical', StandardScaler(),
                                                  ['planned_depth_m',
                                                   'descent_current_speed_m_s',
                                                   'water_temperature_c',
                                                   'sonar_noise_db',
                                                   'navigation_drift_m',
                                                   'battery_health_pct',
                                                   'thruster_vibration_mm_s',
                                                   'missions_since_service']),
                                                 ('nominal',
                                                  OneHotEncoder(handle_unknown='ignore'),
                         

**Analysis:**

### K-nearest neighbours

In [ ]:
neighboursearch = GridSearchCV(estimator=Pipeline(steps=[("preprocess", ct), ("classifier", KNeighborsClassifier())]), param_grid={"n_neighbors": [3, 5, 9, 15, 31, 61, 121], "weights": ["uniform", "distance"]}, scoring="f1", cv=folds, refit=True)

neighboursearch.fit(xtrain, ytrain)

print("Best configuration:", neighboursearch.best_params_)
print("Best mean CV F1:", neighboursearch.best_score_)

**Analysis:**

### Random forest

In [ ]:
forestsearch = GridSearchCV(estimator=Pipeline(steps=[("preprocess", ct), ("classifier", RandomForestClassifier(random_state=random_state))]), param_grid={"n_estimators": [100, 300], "max_depth": [6, 10, 20, None], "min_samples_leaf": [1, 5, 20]}, scoring="f1", cv=folds, refit=True)

forestsearch.fit(xtrain, ytrain)

print("Best configuration:", forestsearch.best_params_)
print("Best mean CV F1:", forestsearch.best_score_)

**Analysis:**

## 2.5 Select and test one final model

Compare the default and tuned results. Select exactly one classifier
family and one associated preprocessing and hyperparameter
configuration; together, these choices are your final pipeline. Record
its mean cross-validation F1 to at least four decimal places before
evaluating it once on the untouched complete test partition. Report
precision, recall, F1, ROC AUC, and a labelled confusion matrix. Compare
cross-validation and test performance, then discuss one limitation of
choosing a model primarily by F1.

In [ ]:
# Your code

**Selection, test results, and interpretation:**

## 2.6 Evaluate the effect of missing values

Summarize missingness by predictor, across all predictor cells, and by
mission. Apply the saved split to the cleaned data. Add the required
imputation inside the pipeline and compare the complete and cleaned
versions using the selected model, fixed hyperparameters, development
rows, and identical folds. Report fold-by-fold differences, computed as
cleaned minus complete, and their means for every required metric.

In [ ]:
# Your code

**Controlled comparison and interpretation:**

## 2.7 Produce predictions for the hidden missions

Load the hidden data into a DataFrame named `hidden_missions`; do not
change this name. Refit the locked complete-data pipeline on all
labelled missions and generate `asrc_missions_hidden_predictions.csv`
with exactly the required columns. Do not round probabilities before
saving them. Validate the mission IDs, classes, class-probability
consistency, and row order. State to at least four decimal places the
cross-validation F1 associated with this exact pipeline. Consult the
handout for the allocation of the two hidden-prediction marks.

In [ ]:
# Your code

The following function validates only the saved file’s format and
internal consistency. It does not generate predictions or provide
information about hidden-set performance.

In [ ]:
import pandas as pd


def validate_hidden_prediction_file(path, hidden_missions):
    predictions = pd.read_csv(path)
    expected_columns = [
        "mission_id",
        "predicted_class",
        "predicted_probability",
    ]

    assert list(predictions.columns) == expected_columns, (
        f"Expected columns {expected_columns}, "
        f"but found {list(predictions.columns)}"
    )
    assert len(predictions) == len(hidden_missions), (
        f"Expected {len(hidden_missions)} rows, "
        f"but found {len(predictions)}"
    )
    assert predictions["mission_id"].is_unique, (
        "Each mission_id must occur exactly once"
    )

    submitted_ids = predictions["mission_id"].reset_index(drop=True)
    expected_ids = hidden_missions["mission_id"].reset_index(drop=True)
    assert submitted_ids.equals(expected_ids), (
        "mission_id values or row order do not match the hidden data"
    )
    assert predictions["predicted_class"].isin([0, 1]).all(), (
        "predicted_class must contain only 0 and 1"
    )

    probabilities = pd.to_numeric(
        predictions["predicted_probability"], errors="coerce"
    )
    assert probabilities.notna().all(), (
        "predicted_probability must be numeric and complete"
    )
    assert probabilities.between(0, 1).all(), (
        "predicted_probability values must lie between 0 and 1"
    )
    expected_classes = (probabilities > 0.5).astype(int)
    assert predictions["predicted_class"].astype(int).equals(
        expected_classes
    ), (
        "predicted_class does not agree with predicted_probability "
        "at the default threshold"
    )

    print("Prediction file structure is valid.")
    return predictions


# Run after saving your prediction file. Do not change the DataFrame name.
# validate_hidden_prediction_file(
#     "asrc_missions_hidden_predictions.csv", hidden_missions
# )

**Final validation:**

## 2.8 Learning curve (optional)

Using only the complete development partition, construct and interpret
the optional learning curve for the selected pipeline.

In [ ]:
# Optional code

**Optional interpretation:**

# References

List all external sources using the IEEE numbered reference style
described in the handout. Cite adapted code in a nearby comment or
Markdown cell. If you did not use external sources, state “No external
sources were used.” You remain responsible for verifying, understanding,
and explaining everything in your submission.

[1] Scikit-learn: Machine Learning in Python, Pedregosa et al., JMLR 12, pp. 2825-2830, 2011.

# AI Use Declaration

**1. Did you use a generative AI tool while completing this
assignment?** This includes assistance with code, debugging, analysis,
interpretation, writing, translation, or finding sources.

**Answer:** No / Yes

If you answered **No**, mark the confirmation below and omit questions 2
through 6 and the AI Interaction Transcript.

- [ ] I confirm that I did not use generative AI in preparing this
  submission.

If you answered **Yes**, complete questions 2 through 6.

**2. Which tools and models or versions did you use?**

**Answer:**

**3. How did you use generative AI?** Select all that apply.

- [ ] Clarifying concepts or checking my understanding
- [ ] Brainstorming or planning an approach
- [ ] Writing or revising code
- [ ] Debugging code or interpreting error messages
- [ ] Interpreting data, results, tables, or plots
- [ ] Writing or revising explanations
- [ ] Translation
- [ ] Finding references or documentation
- [ ] Other:

**4. Briefly identify the sections or tasks for which AI assistance was
used and describe how it influenced your work.**

**Answer:**

**5. How did you verify the AI-generated suggestions, code, or
explanations?**

**Answer:**

**6. Confirmation**

- [ ] I confirm that this declaration is complete, that the required
  transcript is included below, and that I understand and can explain
  everything in my submission.

# AI Interaction Transcript

If you used AI assistance, identify each conversation or session in the
numbered references and include the complete transcript here. If you did
not use AI assistance, omit this section.